In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_table_registry_b (
    table_id STRING,

    table_catalog STRING,
    table_schema STRING,
    table_name STRING,
    fully_qualified_name STRING,

    source_type STRING,
    ingestion_pattern STRING,
    ingestion_frequency STRING,

    partition_column STRING,
    business_date_column STRING,
    expected_arrival_window_minutes INT,

    is_active BOOLEAN,

    registered_timestamp TIMESTAMP,
    registered_by STRING,
    change_reason STRING
)
USING DELTA
COMMENT 'Stores tables onboarded for validation by the Data Trust Layer agent.'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_column_registry_b (
    column_id STRING,

    fully_qualified_name STRING,
    column_name STRING,

    expected_data_type STRING,
    is_nullable BOOLEAN,

    is_active BOOLEAN,
    is_business_key BOOLEAN,

    created_timestamp TIMESTAMP,
    created_by STRING
)
USING DELTA
COMMENT 'Stores columns enrolled for validation for each registered table.'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_column_validation_registry_b (
    column_validation_id STRING,

    fully_qualified_name STRING,
    column_name STRING,
    validation_name STRING,

    is_active BOOLEAN,

    warning_threshold DOUBLE,
    critical_threshold DOUBLE,
    threshold_unit STRING,

    custom_parameters STRING,

    created_timestamp TIMESTAMP,
    created_by STRING
)
USING DELTA
COMMENT 'Stores column-level validation assignments and threshold configurations.'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_table_validation_registry_b (
    table_validation_id STRING,

    fully_qualified_name STRING,
    validation_name STRING,

    is_active BOOLEAN,

    warning_threshold DOUBLE,
    critical_threshold DOUBLE,
    threshold_unit STRING,

    custom_parameters STRING,

    created_timestamp TIMESTAMP,
    created_by STRING
)
USING DELTA
COMMENT 'Stores table-level validation assignments and threshold configurations.'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_baseline_metrics_b (
    baseline_id STRING,

    fully_qualified_name STRING,
    column_name STRING,

    check_type STRING,
    metric_name STRING,

    baseline_date DATE,
    baseline_window_days INT,

    expected_value DOUBLE,

    mean_value DOUBLE,
    stddev_value DOUBLE,
    min_value DOUBLE,
    max_value DOUBLE,
    p25_value DOUBLE,
    p50_value DOUBLE,
    p75_value DOUBLE,
    p95_value DOUBLE,
    sample_count INT,

    is_sufficient_data BOOLEAN,
    baseline_status STRING,

    computed_timestamp TIMESTAMP,
    computed_by STRING
)
USING DELTA
COMMENT 'Stores computed baseline statistics used by the validation agent for anomaly detection.'

In [0]:
%sql

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_validation_run_b (
    validation_run_id STRING,

    fully_qualified_name STRING,
    business_date DATE,
    ingestion_timestamp TIMESTAMP,

    triggered_by STRING,
    pipeline_run_id STRING,

    validation_start_timestamp TIMESTAMP,
    validation_end_timestamp TIMESTAMP,
    validation_duration_seconds DOUBLE,

    overall_status STRING,
    total_checks_run INT,
    total_checks_passed INT,
    total_checks_warned INT,
    total_checks_critical INT,
    total_checks_errored INT,

    ai_enrichment_status STRING,
    ai_enrichment_timestamp TIMESTAMP,

    created_timestamp TIMESTAMP,
    created_by STRING,

    -- Alert tracking (written by pipeline/04_teams_alert_dispatcher)
    alert_sent BOOLEAN,
    alert_sent_timestamp TIMESTAMP,
    alert_channel STRING
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per validation run per table per business date.';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_raw_store_transactions (

    -- Transaction identity
    transaction_id          STRING,
    transaction_date        DATE,
    transaction_timestamp   TIMESTAMP,

    -- Store context
    store_id                STRING,
    store_region            STRING,
    terminal_id             STRING,
    cashier_id              STRING,

    -- Customer context
    customer_id             STRING,
    loyalty_card_number     STRING,

    -- Transaction detail
    transaction_type        STRING,    -- SALE, RETURN, EXCHANGE, VOID
    payment_method          STRING,    -- CASH, CARD, VOUCHER, MOBILE, GIFT_CARD
    channel                 STRING,    -- IN_STORE, CLICK_AND_COLLECT

    -- Financial fields
    gross_amount            DOUBLE,
    discount_amount         DOUBLE,
    tax_amount              DOUBLE,
    net_amount              DOUBLE,
    currency_code           STRING,

    -- Item summary
    item_count              INT,
    quantity_total          INT,

    -- Ingestion metadata
    source_file_name        STRING,
    ingestion_timestamp     TIMESTAMP,
    _rescued_data           STRING

)
USING DELTA
PARTITIONED BY (transaction_date)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_raw_inventory_movements (

    -- Event identity
    event_id                STRING,
    event_timestamp         TIMESTAMP,
    event_type              STRING,    -- RECEIPT, SALE, ADJUSTMENT, TRANSFER, WRITE_OFF

    -- Business date (derived from event_timestamp at load time)
    business_date           DATE,

    -- Location context
    warehouse_id            STRING,
    store_id                STRING,
    location_code           STRING,

    -- Product context
    sku                     STRING,
    product_id              STRING,
    category_code           STRING,
    supplier_id             STRING,

    -- Movement detail
    quantity_change         INT,       -- positive for in, negative for out
    quantity_before         INT,
    quantity_after          INT,
    unit_cost               DOUBLE,
    total_cost_impact       DOUBLE,

    -- Reference
    reference_document_id   STRING,    -- PO number, transfer doc, sale transaction ID
    reference_document_type STRING,    -- PO, TRANSFER, SALE, MANUAL

    -- CDC metadata
    cdc_operation           STRING,    -- INSERT, UPDATE, DELETE
    cdc_source_system       STRING,
    cdc_sequence_number     BIGINT,

    -- Ingestion metadata
    ingestion_timestamp     TIMESTAMP,
    _rescued_data           STRING

)
USING DELTA
PARTITIONED BY (business_date)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc1_validation_results_b (

    -- Identity
    result_id               STRING,
    validation_run_id       STRING,

    -- Scope
    fully_qualified_name    STRING,
    business_date           DATE,

    -- Check identification
    check_type              STRING,
    check_level             STRING,     -- TABLE, COLUMN, PARTITION, PIPELINE
    column_name             STRING,     -- NULL for table-level checks

    -- Deterministic outcome
    check_status            STRING,     -- PASS, INFO, WARNING, CRITICAL, ERROR
    observed_value          DOUBLE,
    baseline_value          DOUBLE,
    deviation_percentage    DOUBLE,
    threshold_warning       DOUBLE,
    threshold_critical      DOUBLE,
    check_detail            STRING,     -- JSON with check-specific metrics
    check_duration_seconds  DOUBLE,

    -- AI enrichment (written asynchronously, NULL until enriched)
    ai_summary              STRING,
    ai_recommended_action   STRING,
    ai_failure_category     STRING,     -- SCHEMA_ISSUE, VOLUME_ISSUE, QUALITY_ISSUE,
                                        -- PIPELINE_ISSUE, COMBINED_ISSUE
    ai_confidence_score     DOUBLE,
    ai_model_version        STRING,
    ai_invocation_id        STRING,

    -- Audit
    created_timestamp       TIMESTAMP,
    ai_enriched_timestamp   TIMESTAMP   -- NULL until AI enrichment completes

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per check per validation run. Stores both deterministic outcomes and async AI enrichment results.'

In [0]:

spark.sql("DELETE FROM learning_dev.s819534.uc1_table_registry_b WHERE true")
spark.sql("DELETE FROM learning_dev.s819534.uc1_column_registry_b WHERE true")
spark.sql("DELETE FROM learning_dev.s819534.uc1_column_validation_registry_b WHERE true")
spark.sql("DELETE FROM learning_dev.s819534.uc1_table_validation_registry_b WHERE true")
spark.sql("DELETE FROM learning_dev.s819534.uc1_baseline_metrics_b WHERE true")



DataFrame[num_affected_rows: bigint]

In [0]:
%sql
-- ── Script 1: Register both tables in uc1_table_registry_b ──────────────────

INSERT INTO learning_dev.s819534.uc1_table_registry_b
(
    table_id,
    table_catalog,
    table_schema,
    table_name,
    fully_qualified_name,
    source_type,
    ingestion_pattern,
    ingestion_frequency,
    partition_column,
    business_date_column,
    expected_arrival_window_minutes,
    is_active,
    registered_timestamp,
    registered_by,
    change_reason
)
VALUES
(
    'tbl-001-store-txn',
    'learning_dev',
    's819534',
    'raw_store_transactions',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'FILE',
    'BATCH',
    'DAILY',
    'transaction_date',
    'transaction_date',
    120,
    true,
    current_timestamp(),
    'agent1_admin',
    'Initial registration for Agent 1 PoC pilot'
),
(
    'tbl-002-inv-mov',
    'learning_dev',
    's819534',
    'raw_inventory_movements',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'CDC',
    'AUTOLOADER',
    'NEAR_REALTIME',
    'business_date',
    'business_date',
    15,
    true,
    current_timestamp(),
    'agent1_admin',
    'Initial registration for Agent 1 PoC pilot'
);

-- Verify
SELECT
    table_id,
    fully_qualified_name,
    source_type,
    ingestion_pattern,
    ingestion_frequency,
    partition_column,
    business_date_column,
    expected_arrival_window_minutes,
    is_active
FROM learning_dev.s819534.uc1_table_registry_b
ORDER BY registered_timestamp;

table_id,fully_qualified_name,source_type,ingestion_pattern,ingestion_frequency,partition_column,business_date_column,expected_arrival_window_minutes,is_active
tbl-001-store-txn,learning_dev.s819534.uc1_raw_store_transactions,FILE,BATCH,DAILY,transaction_date,transaction_date,120,true
tbl-002-inv-mov,learning_dev.s819534.uc1_raw_inventory_movements,CDC,AUTOLOADER,NEAR_REALTIME,business_date,business_date,15,true


In [0]:
%sql
-- ── Script 2: Register columns in uc1_column_registry_b ────────────────────

INSERT INTO learning_dev.s819534.uc1_column_registry_b
(
    column_id,
    fully_qualified_name,
    column_name,
    expected_data_type,
    is_nullable,
    is_active,
    is_business_key,
    created_timestamp,
    created_by
)
VALUES

-- ── raw_store_transactions columns ────────────────────────────────────────

(
    'col-txn-001',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'transaction_id',
    'STRING',
    false,
    true,
    true,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-002',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'transaction_date',
    'DATE',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-003',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'store_id',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-004',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'cashier_id',
    'STRING',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-005',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'customer_id',
    'STRING',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-006',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'transaction_type',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-007',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'payment_method',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-008',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'gross_amount',
    'DOUBLE',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-009',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'discount_amount',
    'DOUBLE',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-010',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'tax_amount',
    'DOUBLE',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-txn-011',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'net_amount',
    'DOUBLE',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),

-- ── raw_inventory_movements columns ──────────────────────────────────────

(
    'col-inv-001',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'event_id',
    'STRING',
    false,
    true,
    true,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-002',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'event_timestamp',
    'TIMESTAMP',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-003',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'event_type',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-004',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'warehouse_id',
    'STRING',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-005',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'store_id',
    'STRING',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-006',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'sku',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-007',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_change',
    'INT',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-008',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_before',
    'INT',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-009',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_after',
    'INT',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-010',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'unit_cost',
    'DOUBLE',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-011',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    '_rescued_data',
    'STRING',
    true,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
),
(
    'col-inv-012',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'cdc_operation',
    'STRING',
    false,
    true,
    false,
    current_timestamp(),
    'agent1_admin'
);

-- Verify
SELECT
    fully_qualified_name,
    column_name,
    expected_data_type,
    is_nullable,
    is_business_key,
    is_active
FROM learning_dev.s819534.uc1_column_registry_b
ORDER BY fully_qualified_name, column_name;

fully_qualified_name,column_name,expected_data_type,is_nullable,is_business_key,is_active
learning_dev.s819534.uc1_raw_inventory_movements,_rescued_data,STRING,true,false,true
learning_dev.s819534.uc1_raw_inventory_movements,cdc_operation,STRING,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,event_id,STRING,false,true,true
learning_dev.s819534.uc1_raw_inventory_movements,event_timestamp,TIMESTAMP,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,event_type,STRING,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,quantity_after,INT,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,quantity_before,INT,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,quantity_change,INT,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,sku,STRING,false,false,true
learning_dev.s819534.uc1_raw_inventory_movements,store_id,STRING,true,false,true


In [0]:
%sql
-- ── Script 3: Table-level validations in uc1_table_validation_registry_b ────

INSERT INTO learning_dev.s819534.uc1_table_validation_registry_b
(
    table_validation_id,
    fully_qualified_name,
    validation_name,
    is_active,
    warning_threshold,
    critical_threshold,
    threshold_unit,
    custom_parameters,
    created_timestamp,
    created_by
)
VALUES

-- ── raw_store_transactions table-level validations ────────────────────────

(
    'tv-txn-001',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'VOLUME_ANOMALY',
    true,
    20.0,
    40.0,
    'PERCENTAGE',
    '{"direction": "both", "baseline_window_days": 30}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-txn-002',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SCHEMA_CONFORMANCE',
    true,
    0.0,
    1.0,
    'COUNT',
    '{"mode": "strict"}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-txn-003',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'PARTITION_FRESHNESS',
    true,
    30.0,
    120.0,
    'MINUTES',
    '{"expected_hour": 2}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-txn-004',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'DUPLICATE_INGESTION',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{"business_key_columns": ["transaction_id"]}',
    current_timestamp(),
    'agent1_admin'
),

-- ── raw_inventory_movements table-level validations ───────────────────────

(
    'tv-inv-001',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'VOLUME_ANOMALY',
    true,
    30.0,
    50.0,
    'PERCENTAGE',
    '{"direction": "both", "baseline_window_days": 30}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-inv-002',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SCHEMA_CONFORMANCE',
    true,
    0.0,
    1.0,
    'COUNT',
    '{"mode": "strict"}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-inv-003',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'PARTITION_FRESHNESS',
    true,
    15.0,
    30.0,
    'MINUTES',
    '{"expected_latency_minutes": 15}',
    current_timestamp(),
    'agent1_admin'
),
(
    'tv-inv-004',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'MALFORMED_RECORDS',
    true,
    1.0,
    5.0,
    'PERCENTAGE',
    '{"indicator_column": "_rescued_data"}',
    current_timestamp(),
    'agent1_admin'
);

-- Verify
SELECT
    fully_qualified_name,
    validation_name,
    is_active,
    warning_threshold,
    critical_threshold,
    threshold_unit,
    custom_parameters
FROM learning_dev.s819534.uc1_table_validation_registry_b
ORDER BY fully_qualified_name, validation_name;

fully_qualified_name,validation_name,is_active,warning_threshold,critical_threshold,threshold_unit,custom_parameters
learning_dev.s819534.uc1_raw_inventory_movements,MALFORMED_RECORDS,true,1.0,5.0,PERCENTAGE,"{""indicator_column"": ""_rescued_data""}"
learning_dev.s819534.uc1_raw_inventory_movements,PARTITION_FRESHNESS,true,15.0,30.0,MINUTES,"{""expected_latency_minutes"": 15}"
learning_dev.s819534.uc1_raw_inventory_movements,SCHEMA_CONFORMANCE,true,0.0,1.0,COUNT,"{""mode"": ""strict""}"
learning_dev.s819534.uc1_raw_inventory_movements,VOLUME_ANOMALY,true,30.0,50.0,PERCENTAGE,"{""direction"": ""both"", ""baseline_window_days"": 30}"
learning_dev.s819534.uc1_raw_store_transactions,DUPLICATE_INGESTION,true,0.1,1.0,PERCENTAGE,"{""business_key_columns"": [""transaction_id""]}"
learning_dev.s819534.uc1_raw_store_transactions,PARTITION_FRESHNESS,true,30.0,120.0,MINUTES,"{""expected_hour"": 2}"
learning_dev.s819534.uc1_raw_store_transactions,SCHEMA_CONFORMANCE,true,0.0,1.0,COUNT,"{""mode"": ""strict""}"
learning_dev.s819534.uc1_raw_store_transactions,VOLUME_ANOMALY,true,20.0,40.0,PERCENTAGE,"{""direction"": ""both"", ""baseline_window_days"": 30}"


In [0]:
%sql
-- ── Script 4: Column-level validations in uc1_column_validation_registry_b ──

INSERT INTO learning_dev.s819534.uc1_column_validation_registry_b
(
    column_validation_id,
    fully_qualified_name,
    column_name,
    validation_name,
    is_active,
    warning_threshold,
    critical_threshold,
    threshold_unit,
    custom_parameters,
    created_timestamp,
    created_by
)
VALUES

-- ── raw_store_transactions column-level validations ───────────────────────

-- transaction_id
(
    'cv-txn-001',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'transaction_id',
    'NULL_SPIKE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- store_id
(
    'cv-txn-002',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'store_id',
    'NULL_SPIKE',
    true,
    1.0,
    3.0,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- customer_id
(
    'cv-txn-003',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'customer_id',
    'NULL_SPIKE',
    true,
    20.0,
    50.0,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- transaction_type
(
    'cv-txn-004',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'transaction_type',
    'CATEGORICAL_DOMAIN',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{"valid_values": ["SALE", "RETURN", "EXCHANGE", "VOID"]}',
    current_timestamp(),
    'agent1_admin'
),

-- payment_method
(
    'cv-txn-005',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'payment_method',
    'CATEGORICAL_DOMAIN',
    true,
    0.5,
    2.0,
    'PERCENTAGE',
    '{"valid_values": ["CASH", "CARD", "VOUCHER", "MOBILE", "GIFT_CARD"]}',
    current_timestamp(),
    'agent1_admin'
),

-- gross_amount: null spike
(
    'cv-txn-006',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'gross_amount',
    'NULL_SPIKE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- gross_amount: numeric range
(
    'cv-txn-007',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'gross_amount',
    'NUMERIC_RANGE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{"min_value": 0.0, "max_value": 50000.0}',
    current_timestamp(),
    'agent1_admin'
),

-- discount_amount: numeric range
(
    'cv-txn-008',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'discount_amount',
    'NUMERIC_RANGE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{"min_value": 0.0, "max_value": 10000.0}',
    current_timestamp(),
    'agent1_admin'
),

-- net_amount: null spike
(
    'cv-txn-009',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'net_amount',
    'NULL_SPIKE',
    true,
    1.0,
    5.0,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- net_amount: numeric range
(
    'cv-txn-010',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'net_amount',
    'NUMERIC_RANGE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{"min_value": -1000.0, "max_value": 50000.0}',
    current_timestamp(),
    'agent1_admin'
),

-- cross-column consistency for transactions
(
    'cv-txn-011',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'net_amount',
    'CROSS_COLUMN_CONSISTENCY',
    true,
    1.0,
    5.0,
    'PERCENTAGE',
    '{"rule": "gross_amount - discount_amount + tax_amount = net_amount", "tolerance": 0.01}',
    current_timestamp(),
    'agent1_admin'
),

-- ── raw_inventory_movements column-level validations ──────────────────────

-- event_id: null spike
(
    'cv-inv-001',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'event_id',
    'NULL_SPIKE',
    true,
    0.0,
    0.1,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- event_type: categorical domain
(
    'cv-inv-002',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'event_type',
    'CATEGORICAL_DOMAIN',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{"valid_values": ["RECEIPT", "SALE", "ADJUSTMENT", "TRANSFER", "WRITE_OFF"]}',
    current_timestamp(),
    'agent1_admin'
),

-- cdc_operation: categorical domain
(
    'cv-inv-003',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'cdc_operation',
    'CATEGORICAL_DOMAIN',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{"valid_values": ["INSERT", "UPDATE", "DELETE"]}',
    current_timestamp(),
    'agent1_admin'
),

-- sku: null spike
(
    'cv-inv-004',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'sku',
    'NULL_SPIKE',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- sku: cardinality drift
(
    'cv-inv-005',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'sku',
    'CARDINALITY_DRIFT',
    true,
    20.0,
    50.0,
    'PERCENTAGE',
    '{"direction": "both"}',
    current_timestamp(),
    'agent1_admin'
),

-- quantity_change: null spike
(
    'cv-inv-006',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_change',
    'NULL_SPIKE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- quantity_after: null spike
(
    'cv-inv-007',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_after',
    'NULL_SPIKE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- quantity_after: numeric range
(
    'cv-inv-008',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_after',
    'NUMERIC_RANGE',
    true,
    0.1,
    1.0,
    'PERCENTAGE',
    '{"min_value": 0, "max_value": 999999}',
    current_timestamp(),
    'agent1_admin'
),

-- unit_cost: null spike
(
    'cv-inv-009',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'unit_cost',
    'NULL_SPIKE',
    true,
    5.0,
    20.0,
    'PERCENTAGE',
    '{}',
    current_timestamp(),
    'agent1_admin'
),

-- unit_cost: numeric range
(
    'cv-inv-010',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'unit_cost',
    'NUMERIC_RANGE',
    true,
    0.1,
    0.5,
    'PERCENTAGE',
    '{"min_value": 0.0, "max_value": 100000.0}',
    current_timestamp(),
    'agent1_admin'
),

-- cross-column consistency for inventory
(
    'cv-inv-011',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'quantity_after',
    'CROSS_COLUMN_CONSISTENCY',
    true,
    1.0,
    5.0,
    'PERCENTAGE',
    '{"rule": "quantity_before + quantity_change = quantity_after", "tolerance": 0}',
    current_timestamp(),
    'agent1_admin'
),

-- _rescued_data: null spike (used as malformed record indicator)
(
    'cv-inv-012',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    '_rescued_data',
    'NULL_SPIKE',
    true,
    1.0,
    5.0,
    'PERCENTAGE',
    '{"invert": true, "description": "flag when non-null rate increases above threshold"}',
    current_timestamp(),
    'agent1_admin'
);

-- Verify
SELECT
    fully_qualified_name,
    column_name,
    validation_name,
    is_active,
    warning_threshold,
    critical_threshold,
    threshold_unit,
    custom_parameters
FROM learning_dev.s819534.uc1_column_validation_registry_b
ORDER BY fully_qualified_name, column_name, validation_name;

fully_qualified_name,column_name,validation_name,is_active,warning_threshold,critical_threshold,threshold_unit,custom_parameters
learning_dev.s819534.uc1_raw_inventory_movements,_rescued_data,NULL_SPIKE,true,1.0,5.0,PERCENTAGE,"{""invert"": true, ""description"": ""flag when non-null rate increases above threshold""}"
learning_dev.s819534.uc1_raw_inventory_movements,cdc_operation,CATEGORICAL_DOMAIN,true,0.1,1.0,PERCENTAGE,"{""valid_values"": [""INSERT"", ""UPDATE"", ""DELETE""]}"
learning_dev.s819534.uc1_raw_inventory_movements,event_id,NULL_SPIKE,true,0.0,0.1,PERCENTAGE,{}
learning_dev.s819534.uc1_raw_inventory_movements,event_type,CATEGORICAL_DOMAIN,true,0.1,1.0,PERCENTAGE,"{""valid_values"": [""RECEIPT"", ""SALE"", ""ADJUSTMENT"", ""TRANSFER"", ""WRITE_OFF""]}"
learning_dev.s819534.uc1_raw_inventory_movements,quantity_after,CROSS_COLUMN_CONSISTENCY,true,1.0,5.0,PERCENTAGE,"{""rule"": ""quantity_before + quantity_change = quantity_after"", ""tolerance"": 0}"
learning_dev.s819534.uc1_raw_inventory_movements,quantity_after,NULL_SPIKE,true,0.1,0.5,PERCENTAGE,{}
learning_dev.s819534.uc1_raw_inventory_movements,quantity_after,NUMERIC_RANGE,true,0.1,1.0,PERCENTAGE,"{""min_value"": 0, ""max_value"": 999999}"
learning_dev.s819534.uc1_raw_inventory_movements,quantity_change,NULL_SPIKE,true,0.1,0.5,PERCENTAGE,{}
learning_dev.s819534.uc1_raw_inventory_movements,sku,CARDINALITY_DRIFT,true,20.0,50.0,PERCENTAGE,"{""direction"": ""both""}"
learning_dev.s819534.uc1_raw_inventory_movements,sku,NULL_SPIKE,true,0.1,1.0,PERCENTAGE,{}


In [0]:
%sql
-- ── Verification 1: Table registration summary ────────────────────────────

SELECT
    fully_qualified_name,
    source_type,
    ingestion_pattern,
    ingestion_frequency,
    partition_column,
    business_date_column,
    expected_arrival_window_minutes,
    is_active
FROM learning_dev.s819534.uc1_table_registry_b
ORDER BY fully_qualified_name;


-- ── Verification 2: Column registration summary ───────────────────────────

SELECT
    fully_qualified_name,
    COUNT(*)                                            AS total_columns,
    SUM(CASE WHEN is_business_key  THEN 1 ELSE 0 END)  AS business_key_columns,
    SUM(CASE WHEN is_nullable      THEN 1 ELSE 0 END)  AS nullable_columns,
    SUM(CASE WHEN is_active        THEN 1 ELSE 0 END)  AS active_columns
FROM learning_dev.s819534.uc1_column_registry_b
GROUP BY fully_qualified_name
ORDER BY fully_qualified_name;


-- ── Verification 3: Table validation summary ──────────────────────────────

SELECT
    fully_qualified_name,
    COUNT(*)                                           AS total_validations,
    SUM(CASE WHEN is_active THEN 1 ELSE 0 END)        AS active_validations,
    COLLECT_LIST(validation_name)                      AS validations_configured
FROM learning_dev.s819534.uc1_table_validation_registry_b
GROUP BY fully_qualified_name
ORDER BY fully_qualified_name;


-- ── Verification 4: Column validation summary ─────────────────────────────

SELECT
    fully_qualified_name,
    column_name,
    COLLECT_LIST(validation_name)                      AS validations,
    COUNT(*)                                           AS validation_count
FROM learning_dev.s819534.uc1_column_validation_registry_b
GROUP BY fully_qualified_name, column_name
ORDER BY fully_qualified_name, column_name;


-- ── Verification 5: Complete configuration view ───────────────────────────

SELECT
    t.fully_qualified_name,
    t.source_type,
    t.ingestion_pattern,
    COUNT(DISTINCT cr.column_name)                     AS enrolled_columns,
    COUNT(DISTINCT tv.validation_name)                 AS table_validations,
    COUNT(DISTINCT cv.column_validation_id)            AS column_validations,
    COUNT(DISTINCT cv.column_validation_id) +
    COUNT(DISTINCT tv.table_validation_id)             AS total_checks_configured
FROM learning_dev.s819534.uc1_table_registry_b t
LEFT JOIN learning_dev.s819534.uc1_column_registry_b cr
    ON  t.fully_qualified_name = cr.fully_qualified_name
    AND cr.is_active = true
LEFT JOIN learning_dev.s819534.uc1_table_validation_registry_b tv
    ON  t.fully_qualified_name = tv.fully_qualified_name
    AND tv.is_active = true
LEFT JOIN learning_dev.s819534.uc1_column_validation_registry_b cv
    ON  t.fully_qualified_name = cv.fully_qualified_name
    AND cv.is_active = true
GROUP BY
    t.fully_qualified_name,
    t.source_type,
    t.ingestion_pattern
ORDER BY t.fully_qualified_name;

fully_qualified_name,source_type,ingestion_pattern,enrolled_columns,table_validations,column_validations,total_checks_configured
comms_media_dev.exposure_data_asset.bronze_barrows_campaign,FILE,BRONZE,13,3,48,51
learning_dev.s819534.uc1_raw_inventory_movements,CDC,AUTOLOADER,12,4,12,16
learning_dev.s819534.uc1_raw_store_transactions,FILE,BATCH,11,4,11,15
